# FHB Severity Estimation — Combined Multi-Method Comparison (M1 + SAM3)

Runs **one** detection + segmentation pass per image, then applies **all five**
pixel-classification methods to the same wheat-head masks and writes a single
Excel comparison file.

**Pipeline**
1. Detect wheat heads with model — `max_det=800`
2. Segment every detected head with **SAM3** in one batched call; masks are pulled
   off-device **one at a time** to keep memory flat on dense plots.
3. For each head mask, classify its RGB pixels with each method and take
   `% pixels classified Infected` as that head's severity:
   - **Index** — Red/Green ratio threshold `R/(R+G) > 0.5`
   - **GMM** — 2-component full-covariance RGB GMM fit *per image* on all masked
     pixels, initialised from `means_init.csv` / `precisions_init.csv`
   - **BayesGMM** — per-class GMM density estimators + Bayes rule (prior 0.5)
   - **LogReg** — standardised logistic regression trained on `Pixels_All.xlsx`
   - **SVM** — grid-searched SVC trained on `Pixels_All.xlsx`
4. Average each method's per-head severity across all heads in the image.
5. Save one row per image to excel file.

Output columns: `Plot | Num_Heads | FHB_SEV_Index | FHB_SEV_GMM | FHB_SEV_BayesGMM | FHB_SEV_LogReg | FHB_SEV_SVM`

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt

from ultralytics import YOLO, SAM
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import classification_report

np.random.seed(42)

## 1. Configuration — set paths here
Replace all with local paths

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ROOT_DIR    = #path to images folder          # folder of images to process
OUTPUT_XLSX = # path to output xlsx file          # output Excel file for pixel-level results

M1_WEIGHTS      = 'Model_Weights\M1.pt'          # YOLO detector, GWHD-only
M2_WEIGHTS      = 'Model_Weights\M2.pt'          # YOLO detector, GWHD + Elora
M3_WEIGHTS      = 'Model_Weights\M3.pt'          # YOLO detector, GWHD + Elora + null
M4_WEIGHTS      = 'Model_Weights\M4.pt'          # YOLO detector, GWHD + Elora oversampled + null
M5_WEIGHTS      = 'Model_Weights\M5.pt'          # YOLO detector, M1 fine-tuned on Elora
M6_WEIGHTS      = 'Model_Weights\M6.pt'          # YOLO detector, Elora-only 
SAM3_CHECKPOINT = 'sam3.pt'

PIXEL_VALUES_XLSX  = 'Data\Pixels_All.xlsx'   # R,G,B,Cluster training pixels
GMM_MEANS_CSV      = 'Data\means_init.csv'
GMM_PRECISIONS_CSV = 'Data\precisions_init.csv'

# ── Detection ─────────────────────────────────────────────────────────────
MAX_DET = 800          # matches M1 training; avoids capping dense plots. No conf override (YOLO default).

# ── Method settings ─────────────────────────────────────────────────────────
RG_THRESHOLD   = 0.5    # Index: R/(R+G) ratio above which a pixel counts as infected
PRIOR_INFECTED = 0.5    # BayesGMM: prior P(Infected); PRIOR_HEALTHY = 1 - PRIOR_INFECTED
N_COMPONENTS   = 2      # BayesGMM: components per per-class GMM
N_FOLDS        = 5      # LogReg / SVM: stratified k-folds for the reported CV
LR_C           = 1.0    # LogReg: inverse regularisation strength

os.makedirs(os.path.dirname(OUTPUT_XLSX), exist_ok=True)

## 2. Load Pixel Training Data

`Pixels_All.xlsx` holds labelled RGB pixels used by **BayesGMM**, **LogReg**
and **SVM**. `Cluster` is `0 = Healthy`, `1 = Infected`.

In [ ]:
df_vals = pd.read_excel(PIXEL_VALUES_XLSX, sheet_name='Sheet1')
print(f'Pixel training data shape: {df_vals.shape}')
print('Class balance:')
print(df_vals['Cluster'].value_counts().rename({0: 'Healthy', 1: 'Infected'}))

X = df_vals[['R', 'G', 'B']].values.astype(np.float64)
y = df_vals['Cluster'].values

## 3. Fit / Load the Fixed Classifiers

Everything here is computed once and reused for every image.

* **BayesGMM** — one GMM density estimator per class.
* **LogReg** — standardise → logistic regression, with a reported k-fold CV.
* **SVM** — grid-searched SVC, with a reported k-fold CV, refit on all data.
* **GMM inits** — `means_init` / `precisions_init` seed the *per-image* GMM fit
  done later inside the loop.

In [ ]:
# ── BayesGMM: per-class density estimators ────────────────────────────────
gmm_healthy  = GaussianMixture(n_components=N_COMPONENTS, covariance_type='full', random_state=42)
gmm_infected = GaussianMixture(n_components=N_COMPONENTS, covariance_type='full', random_state=42)
gmm_healthy.fit(X[y == 0])
gmm_infected.fit(X[y == 1])
prior_healthy = 1 - PRIOR_INFECTED
print('BayesGMM Healthy means:\n', pd.DataFrame(gmm_healthy.means_, columns=['R', 'G', 'B']))
print('BayesGMM Infected means:\n', pd.DataFrame(gmm_infected.means_, columns=['R', 'G', 'B']))

In [ ]:
# ── LogReg: standardise + logistic regression ────────────────────────────
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(C=LR_C, max_iter=1000, random_state=42)),
])

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
lr_cv = cross_validate(logreg_pipeline, X, y, cv=cv,
                       scoring=['accuracy', 'precision', 'recall', 'f1'])
print(f'=== LogReg {N_FOLDS}-Fold CV ===')
for m in ['accuracy', 'precision', 'recall', 'f1']:
    s = lr_cv[f'test_{m}']
    print(f'  {m:>10}: {s.mean():.4f} +/- {s.std():.4f}')

logreg_pipeline.fit(X, y)
print('\nLogReg trained on full pixel dataset.')
print(classification_report(y, logreg_pipeline.predict(X), target_names=['Healthy', 'Infected']))

In [ ]:
# ── SVM: grid-searched SVC ──────────────────────────────────────────────
param_grid = {
    'svm__C':      [0.1, 1, 10, 100],
    'svm__gamma':  ['scale', 'auto', 0.001, 0.01, 0.1],
    'svm__kernel': ['rbf', 'linear'],
}
grid_pipeline = Pipeline([('scaler', StandardScaler()), ('svm', SVC(random_state=42))])
grid_search = GridSearchCV(grid_pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X, y)
print(f'\nBest params : {grid_search.best_params_}')
print(f'Best CV F1   : {grid_search.best_score_:.4f}')

best = grid_search.best_params_
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(C=best['svm__C'], gamma=best['svm__gamma'], kernel=best['svm__kernel'], random_state=42)),
])
svm_cv = cross_validate(svm_pipeline, X, y, cv=cv,
                        scoring=['accuracy', 'precision', 'recall', 'f1'])
print(f'\n=== SVM {N_FOLDS}-Fold CV (best params) ===')
for m in ['accuracy', 'precision', 'recall', 'f1']:
    s = svm_cv[f'test_{m}']
    print(f'  {m:>10}: {s.mean():.4f} +/- {s.std():.4f}')

svm_pipeline.fit(X, y)
print('\nSVM trained on full pixel dataset.')
print(classification_report(y, svm_pipeline.predict(X), target_names=['Healthy', 'Infected']))

In [ ]:
# ── GMM initialisations for the per-image fit ───────────────────────────
means_df = pd.read_csv(GMM_MEANS_CSV,      index_col=0)
prec_df  = pd.read_csv(GMM_PRECISIONS_CSV, index_col=0)

means_init      = means_df[['R', 'G', 'B']].values.astype(np.float64)               # (2, 3): row 0 Healthy, row 1 Infected
prec_flat       = prec_df[['R_R', 'R_G', 'R_B', 'G_R', 'G_G', 'G_B',
                           'B_R', 'B_G', 'B_B']].values.astype(np.float64)
precisions_init = prec_flat.reshape(2, 3, 3)                                         # (2, 3, 3)

print('GMM means_init:\n', means_df)
print(f'\nprecisions_init shape: {precisions_init.shape}')

## 4. Load Detection and Segmentation Models

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

yolo_M1 = YOLO(M1_WEIGHTS)     # M1 — YOLO trained on GWHD only
yolo_M2 = YOLO(M2_WEIGHTS)     # M2 — YOLO trained on GWHD + Elora
yolo_M3 = YOLO(M3_WEIGHTS)     # M3 — YOLO trained on GWHD + Elora + null
yolo_M4 = YOLO(M4_WEIGHTS)     # M4 — YOLO trained on GWHD + Elora oversampled + null
yolo_M5 = YOLO(M5_WEIGHTS)     # M5 — YOLO fine-tuned on Elora
yolo_M6 = YOLO(M6_WEIGHTS)     # M6 — YOLO trained on Elora only
sam3_model = SAM(SAM3_CHECKPOINT)

print('Models loaded.')

## 5. Fused Processing Loop

For every image: **M1 detect once → SAM3 segment once → all five methods on the
same masks.**

`Num_Heads` is therefore identical across methods. A method only diverges when
its own fit fails (per-image **GMM**) or a mask is empty (that head is skipped
for every method so the head count stays consistent).

The per-image **GMM** needs all masked pixels together, so masks are walked
twice: pass 1 builds the combined mask and fits the GMM; pass 2 classifies each
head with every method. Both passes pull masks off the GPU **one slice at a
time**.

In [ ]:
records = []
image_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

METHOD_COLS = ['FHB_SEV_Index', 'FHB_SEV_GMM', 'FHB_SEV_BayesGMM', 'FHB_SEV_LogReg', 'FHB_SEV_SVM']

for dirpath, _, filenames in os.walk(ROOT_DIR):
    for filename in sorted(filenames):
        if not filename.lower().endswith(image_extensions):
            continue

        img_path = os.path.join(dirpath, filename)
        plot_id  = os.path.splitext(filename)[0]
        print(f'Processing: {img_path}', end=' ... ')

        # ── Load image ────────────────────────────────────────────────────
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print('SKIPPED (unreadable)')
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # ── M1 detection (YOLO default conf, max_det=800) ────────────────
        yolo_results = yolo_M6(img_path, max_det=MAX_DET, verbose=False)
        boxes = []
        for result in yolo_results:
            for box in result.boxes.xyxy:
                boxes.append(box.tolist())

        if len(boxes) == 0:
            print('SKIPPED (no detections)')
            records.append({'Plot': plot_id, 'Num_Heads': 0,
                            **{c: np.nan for c in METHOD_COLS}})
            continue

        # ── SAM3 segmentation — one batched call, masks kept on device ───
        sam3_results = sam3_model(img_path, bboxes=boxes, verbose=False)
        masks_tensor = sam3_results[0].masks.data
        n_masks = masks_tensor.shape[0]

        # ── Pass 1: combined mask + per-image GMM fit ────────────────────
        combined_mask = np.zeros(img_rgb.shape[:2], dtype=bool)
        for i in range(n_masks):
            combined_mask |= masks_tensor[i].cpu().numpy().astype(bool)

        all_pixels = img_rgb[combined_mask].astype(np.float64)   # (N, 3)
        gmm = None
        infected_comp = 1
        try:
            gmm = GaussianMixture(
                n_components=2, covariance_type='full',
                means_init=means_init, precisions_init=precisions_init,
                random_state=42, tol=1e-6, max_iter=500,
            )
            gmm.fit(all_pixels)
            d0_healthy  = np.linalg.norm(gmm.means_[0] - means_init[0])
            d0_infected = np.linalg.norm(gmm.means_[0] - means_init[1])
            infected_comp = 1 if d0_healthy < d0_infected else 0
        except Exception as e:
            print(f'[GMM fit failed: {e}] ', end='')
            gmm = None

        # ── Pass 2: classify every head with every method ────────────────
        sev = {c: [] for c in METHOD_COLS}
        for i in range(n_masks):
            mask = masks_tensor[i].cpu().numpy().astype(bool)
            pixels = img_rgb[mask].astype(np.float64)   # (M, 3)
            if pixels.shape[0] == 0:
                continue

            # Index — Red/Green ratio
            r, g = pixels[:, 0], pixels[:, 1]
            rg = r / (r + g)
            sev['FHB_SEV_Index'].append(float(np.sum(rg > RG_THRESHOLD) / pixels.shape[0] * 100))

            # GMM — per-image fit
            if gmm is None:
                sev['FHB_SEV_GMM'].append(np.nan)
            else:
                raw = gmm.predict(pixels)
                labels = raw if infected_comp == 1 else 1 - raw
                sev['FHB_SEV_GMM'].append(float(np.sum(labels == 1) / len(labels) * 100))

            # BayesGMM — per-class density + Bayes rule
            lp_healthy  = gmm_healthy.score_samples(pixels)  + np.log(prior_healthy)
            lp_infected = gmm_infected.score_samples(pixels) + np.log(PRIOR_INFECTED)
            b_labels = (lp_infected > lp_healthy).astype(int)
            sev['FHB_SEV_BayesGMM'].append(float(np.sum(b_labels == 1) / len(b_labels) * 100))

            # LogReg
            lr_labels = logreg_pipeline.predict(pixels)
            sev['FHB_SEV_LogReg'].append(float(np.sum(lr_labels == 1) / len(lr_labels) * 100))

            # SVM
            svm_labels = svm_pipeline.predict(pixels)
            sev['FHB_SEV_SVM'].append(float(np.sum(svm_labels == 1) / len(svm_labels) * 100))

        del sam3_results, masks_tensor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        n_heads = len(sev['FHB_SEV_Index'])
        row = {'Plot': plot_id, 'Num_Heads': n_heads}
        for c in METHOD_COLS:
            row[c] = float(np.mean(sev[c])) if len(sev[c]) else np.nan
        records.append(row)

        print(f"{n_heads} heads | " +
              " ".join(f"{c.replace('FHB_SEV_', '')}={row[c]:.1f}%" for c in METHOD_COLS))

# ── Save results ─────────────────────────────────────────────────────────
df_results = pd.DataFrame(records, columns=['Plot', 'Num_Heads'] + METHOD_COLS)
df_results.to_excel(OUTPUT_XLSX, index=False)
print(f'\nDone. Results saved to: {OUTPUT_XLSX}')
df_results

## 6. Summary

In [ ]:
print('=== Results Summary ===')
print(f'Images processed  : {len(df_results)}')
print(f'Images with heads : {df_results["Num_Heads"].gt(0).sum()}')
print(f'Total heads       : {int(df_results["Num_Heads"].sum())}')
print()
print('Per-method mean FHB severity across images:')
for c in METHOD_COLS:
    col = df_results[c].dropna()
    print(f'  {c.replace("FHB_SEV_", ""):>9}: {col.mean():6.2f}%  (std {col.std():5.2f}, n={len(col)})')

print()
print(df_results.to_string(index=False))

# ── Comparison plots ────────────────────────────────────────────────────
valid = df_results.dropna(subset=METHOD_COLS)
if len(valid):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].boxplot([valid[c] for c in METHOD_COLS],
                    labels=[c.replace('FHB_SEV_', '') for c in METHOD_COLS])
    axes[0].set_ylabel('Mean FHB Severity per Image (%)')
    axes[0].set_title('FHB Severity by Method — M1 + SAM3')
    axes[0].grid(axis='y', alpha=0.3)

    for c in METHOD_COLS:
        axes[1].plot(valid['Plot'], valid[c], marker='o', label=c.replace('FHB_SEV_', ''))
    axes[1].set_xlabel('Plot')
    axes[1].set_ylabel('Mean FHB Severity (%)')
    axes[1].set_title('Per-Plot Severity by Method')
    axes[1].tick_params(axis='x', rotation=90)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()